# [Traces] T4 + Qwen3-8B + AIME-Val: Strategy Validation

## Purpose
**Validate** that strategies tuned on Qwen3-4B traces **transfer** to a stronger model.
Uses AIME 2020-2024 as a held-out validation set.

## Why This Configuration?
- **T4 GPU**: Still fits in free quota alongside the 4B bulk job
- **Qwen3-8B**: 2x stronger than 4B, tests strategy transfer
- **AIME 2020-2024**: ~300 problems held out from training

## Pipeline Position
```
1. Qwen3-4B + AIME 1983-2019  → Tune strategies (11K traces)
2. Qwen3-8B + AIME 2020-2024  → THIS: Validate transfer (1.5K traces)
3. Qwen3-30B + AIMO3 ref      → Check competition-level transfer
4. gpt-oss-120b + AIMO3 ref   → Final tuning on exact model
```

## Output
~300 problems × 5 samples = **1,500 traces**
Runtime: ~6-8 hours on T4

In [ ]:
%pip install -q vllm transformers accelerate

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os, sys, json, re, math, time, subprocess, tempfile
from collections import Counter
from typing import Optional, Dict, List, Any
import pandas as pd

os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

In [ ]:
class CFG:
    # Model - Qwen3-8B for validation
    model_name = 'Qwen/Qwen3-8B'
    model_path = '/kaggle/input/qwen-3/transformers/qwen3-8b/1'
    
    system_prompt = (
        'You are a math competition solver. Solve the problem step by step. '
        'You can use Python code in ```python``` blocks which will be executed. '
        'Put your final integer answer in \\boxed{}.'
    )
    
    n_samples = 5
    max_turns = 8
    max_tokens = 4096
    temperature = 0.8
    top_p = 0.95
    
    gpu_memory_utilization = 0.90
    max_model_len = 8192
    
    problem_timeout = 240  # Slightly longer for bigger model
    code_timeout = 10
    
    # Filter to validation years
    val_years = list(range(2020, 2025))  # 2020-2024
    
    output_dir = '/kaggle/working/traces'
    seed = 42

print(f"Config: {CFG.model_name}, validation years: {CFG.val_years}")

In [ ]:
def load_aime_validation():
    """Load only validation years from AIME."""
    path = '/kaggle/input/aime-problem-set-1983-2024/AIME_Dataset_1983_2024.csv'
    df = pd.read_csv(path)
    df.columns = df.columns.str.lower().str.strip()
    
    # Filter to validation years
    if 'year' in df.columns:
        df = df[df['year'].isin(CFG.val_years)]
    
    # Create ID
    if 'id' not in df.columns:
        df['id'] = df.apply(lambda r: f"aime_{r.get('year', 0)}_{r.get('problem number', r.name)}", axis=1)
    
    prob_col = next((c for c in df.columns if 'problem' in c.lower() and 'number' not in c.lower()), None)
    if prob_col and prob_col != 'problem':
        df['problem'] = df[prob_col]
    
    ans_col = next((c for c in df.columns if 'answer' in c.lower()), None)
    if ans_col and ans_col != 'answer':
        df['answer'] = df[ans_col]
    
    print(f"Loaded {len(df)} AIME validation problems (years {CFG.val_years})")
    return df.reset_index(drop=True)

df = load_aime_validation()

In [ ]:
def execute_code(code: str, timeout: int = 10) -> str:
    full_code = "import math, itertools, functools\nfrom fractions import Fraction\n" + code
    try:
        with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
            f.write(full_code)
            f.flush()
            result = subprocess.run(['python', f.name], capture_output=True, text=True, timeout=timeout)
            os.unlink(f.name)
            if result.returncode != 0:
                return f"[ERROR] {result.stderr[:500]}"
            return result.stdout[:2000] or "[No output]"
    except subprocess.TimeoutExpired:
        return "[ERROR] Timeout"
    except Exception as e:
        return f"[ERROR] {str(e)[:200]}"

In [ ]:
from vllm import LLM, SamplingParams

model_path = CFG.model_path if os.path.exists(CFG.model_path) else CFG.model_name
print(f"Loading: {model_path}")

llm = LLM(
    model=model_path,
    gpu_memory_utilization=CFG.gpu_memory_utilization,
    max_model_len=CFG.max_model_len,
    trust_remote_code=True,
    seed=CFG.seed,
)
print("Model loaded")

In [ ]:
def extract_answer(text: str) -> Optional[int]:
    for pattern in [r'\\boxed\s*\{\s*([0-9]+)\s*\}', r'answer\s*(?:is|=)\s*([0-9]+)']:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            try:
                val = int(matches[-1])
                if 0 <= val <= 999:
                    return val
            except: pass
    return None

def extract_code_blocks(text: str) -> List[str]:
    return re.findall(r'```(?:python)?\s*\n(.*?)```', text, re.DOTALL | re.IGNORECASE)

def compute_entropy(logprobs: List[Dict]) -> float:
    if not logprobs: return float('inf')
    total, count = 0.0, 0
    for lp_dict in logprobs:
        if isinstance(lp_dict, dict) and lp_dict:
            ent = sum(-math.exp(lp) * math.log2(max(math.exp(lp), 1e-10)) for lp in lp_dict.values() if lp is not None)
            total += ent
            count += 1
    return total / count if count else float('inf')

In [ ]:
def solve_once(problem_text: str, seed: int) -> Dict[str, Any]:
    messages = [{"role": "system", "content": CFG.system_prompt}, {"role": "user", "content": problem_text}]
    tokenizer = llm.get_tokenizer()
    
    all_logprobs, code_executions = [], []
    answer, answer_source, last_code_output = None, None, None
    turns_used, total_tokens = 0, 0
    
    sampling_params = SamplingParams(temperature=CFG.temperature, top_p=CFG.top_p, max_tokens=CFG.max_tokens, seed=seed, logprobs=5)
    
    for turn in range(CFG.max_turns):
        turns_used = turn + 1
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        outputs = llm.generate([prompt], sampling_params)
        response = outputs[0].outputs[0]
        text = response.text
        total_tokens += len(response.token_ids)
        
        if response.logprobs:
            for lp in response.logprobs:
                if lp: all_logprobs.append({k: v.logprob for k, v in lp.items()})
        
        messages.append({"role": "assistant", "content": text})
        
        ans = extract_answer(text)
        if ans is not None:
            answer, answer_source = ans, "boxed"
            break
        
        code_blocks = extract_code_blocks(text)
        if code_blocks:
            outputs_list = []
            for code in code_blocks:
                output = execute_code(code, CFG.code_timeout)
                is_error = '[ERROR]' in output
                if not is_error: last_code_output = output
                outputs_list.append(output)
                code_executions.append({'turn': turn, 'code': code[:500], 'output': output[:500], 'is_error': is_error})
            messages.append({"role": "user", "content": f"Code output:\n```\n{''.join(outputs_list)}\n```\nContinue. Put answer in \\boxed{{}}." })
        else:
            messages.append({"role": "user", "content": "Continue. Put answer in \\boxed{}."})
    
    if answer is None and last_code_output:
        for num in re.findall(r'\b(\d{1,3})\b', last_code_output):
            val = int(num)
            if 0 <= val <= 999:
                answer, answer_source = val, "code_fallback"
                break
    
    entropy = compute_entropy(all_logprobs)
    if answer_source == "code_fallback": entropy = max(entropy, 8.0)
    
    return {'answer': answer, 'answer_source': answer_source, 'entropy': entropy, 'turns_used': turns_used,
            'total_tokens': total_tokens, 'code_executions': code_executions,
            'n_python_calls': len(code_executions), 'n_python_errors': sum(1 for c in code_executions if c['is_error'])}

In [ ]:
def generate_all_traces(df):
    os.makedirs(CFG.output_dir, exist_ok=True)
    
    config = {'model': CFG.model_name, 'n_samples': CFG.n_samples, 'val_years': CFG.val_years,
              'n_problems': len(df), 'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')}
    with open(f"{CFG.output_dir}/config.json", 'w') as f: json.dump(config, f, indent=2)
    
    all_results, total_correct, start_time = [], 0, time.time()
    
    for idx, row in df.iterrows():
        prob_id = str(row.get('id', idx))
        ground_truth = int(row['answer']) if pd.notna(row.get('answer')) else None
        
        print(f"\nProblem {idx+1}/{len(df)} [{prob_id}] (GT={ground_truth})")
        
        attempts = []
        for i in range(CFG.n_samples):
            seed = CFG.seed + idx * 100 + i * 7
            t0 = time.time()
            try:
                result = solve_once(row['problem'], seed)
                result.update({'attempt_idx': i, 'seed': seed, 'wall_time_s': round(time.time()-t0, 2), 'prompt_type': 'reasoning'})
            except Exception as e:
                result = {'attempt_idx': i, 'answer': None, 'entropy': float('inf'), 'error': str(e)[:200]}
            attempts.append(result)
            print(f"  {i+1}/{CFG.n_samples}: ans={result.get('answer')}, ent={result.get('entropy', 0):.3f}")
        
        valid = [a['answer'] for a in attempts if a['answer'] is not None]
        default_answer = Counter(valid).most_common(1)[0][0] if valid else 0
        is_correct = default_answer == ground_truth if ground_truth is not None else None
        if is_correct: total_correct += 1
        
        print(f"  >> {'✓' if is_correct else '✗'} Default={default_answer}")
        
        trace = {'problem_id': prob_id, 'problem_text': row['problem'], 'ground_truth': ground_truth,
                 'attempts': attempts, 'default_answer': default_answer, 'default_votes': dict(Counter(valid))}
        with open(f"{CFG.output_dir}/problem_{prob_id}.json", 'w') as f:
            json.dump(trace, f, indent=2, default=lambda x: str(x) if isinstance(x, float) and math.isinf(x) else x)
        
        all_results.append({'problem_id': prob_id, 'correct': is_correct})
    
    total = sum(1 for r in all_results if r['correct'] is not None)
    summary = {'model': CFG.model_name, 'correct': total_correct, 'total': total,
               'accuracy': round(total_correct/total, 4) if total else 0, 'total_time_s': round(time.time()-start_time, 1)}
    with open(f"{CFG.output_dir}/summary.json", 'w') as f: json.dump(summary, f, indent=2)
    
    print(f"\nCOMPLETE: {total_correct}/{total} ({summary['accuracy']*100:.1f}%)")
    return summary

summary = generate_all_traces(df)

In [ ]:
print(f"\nValidation Results ({CFG.model_name}):")
print(f"  Accuracy: {summary['accuracy']*100:.1f}%")
print(f"  Time: {summary['total_time_s']/3600:.1f} hours")
print(f"  Traces: {CFG.output_dir}")